In [ ]:
import hashlib, urllib.request
from pathlib import Path
import pandas as pd

CORPUS_URL = "https://github.com/CASDAV/proyecto-pln/releases/download/corpus-v1/noticias_colombia.parquet"
SHA256 = "92b02aa1a74015192d583cede59c3d8bd7794c214028809cc687704255aa2fc7"
DEST = Path("/content/data/noticias_colombia.parquet")


def corpus_path() -> Path:
    if DEST.exists():
        return DEST
    DEST.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CORPUS_URL, DEST)
    got = hashlib.sha256(DEST.read_bytes()).hexdigest()
    if got != SHA256:
        DEST.unlink()
        raise RuntimeError(f"checksum no coincide: {got}")
    return DEST


# --- Estado heredado de 01-corpus: deduplicación ---
df_raw = pd.read_parquet(corpus_path())

norm = (df_raw["texto"].str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip())

df = df_raw[~norm.duplicated()].reset_index(drop=True)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

assert len(df) == 91585, f"esperaba 91585, hay {len(df)}"
print(len(df_raw), "→", len(df))